# 05 - Write HDF5 Outputs

Write selected IceTray keys and a small custom event table to HDF5. HDF5 is convenient for plotting; `.i3` remains the native event format.


In [ ]:
from pathlib import Path
import os
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
from icecube import dataio
from I3Tray import I3Tray
from icecube.hdfwriter import I3HDFWriter

from icetray_tutorial.paths import EXPERIMENT
from icetray_tutorial.frames import event_header_dict, iter_frames, stop_name
from icetray_tutorial.pulses import summarize_pulses


In [ ]:
username = os.environ.get('USER', 'icecube_user')
output_dir = Path(f'/data/user/{username}/IceTray_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)
output_dir


In [ ]:
hdf_output = output_dir / 'level2_small_selection.hdf5'
tray = I3Tray()
tray.AddModule('I3Reader', 'reader', FilenameList=[str(EXPERIMENT.gcd), str(EXPERIMENT.event_file)])
tray.AddSegment(I3HDFWriter, 'hdf_writer', Output=str(hdf_output), Keys=['I3EventHeader', 'QFilterMask'], SubEventStreams=['InIceSplit'])
tray.Execute(500)
tray.Finish()
hdf_output


In [ ]:
rows = []
physics_seen = 0
for frame in iter_frames(EXPERIMENT.event_file):
    if stop_name(frame) not in ('Physics', 'P'):
        continue
    physics_seen += 1
    row = event_header_dict(frame)
    pulse_summary = summarize_pulses(frame)
    if pulse_summary:
        row.update({'pulse_key': pulse_summary.pulse_key, 'hit_doms': pulse_summary.hit_doms, 'pulses': pulse_summary.pulses, 'total_charge': pulse_summary.total_charge})
    rows.append(row)
    if physics_seen >= 1000:
        break

event_table = pd.DataFrame(rows)
event_table.head()


In [ ]:
custom_output = output_dir / 'event_hit_summary.h5'
event_table.to_hdf(custom_output, key='events', mode='w')
pd.read_hdf(custom_output, key='events').head()


## Practice

1. Add `TutorialHitDOMs` in a tray and include it in HDF5 output.
2. Try adding a reconstruction key such as `SplineMPE` if it exists.
3. Save plots made from the HDF5 table under `/data/user/<username>/IceTray_tutorial/figures`.
